# 07 — Another language, another domain

The library's rule about language is short (CLAUDE.md §12): **English in the code,
Portuguese in the patterns.** Names, docstrings and messages are English so the
library is usable outside Brazil; the regexes and word lists stay Portuguese
because they *describe the corpus* — the conformity stamp, the enclitic pronouns,
the forensic abbreviations, the identity-card markers, the legal vocabulary.

That is the adaptation seam, and until the pattern catalogue existed it was a
promise the code could not honour. The patterns lived as `re.compile` calls across
ten modules, so "adapting the library to another corpus is a matter of swapping the
patterns" actually meant editing Python in ten files and hoping the test suite
noticed.

Now they are **data**: TOML, versioned with the package, layered.

```
base.toml     nothing that describes a language — control bytes, glyph-index
              runs, digit runs, markdown structure
pt_br.toml    the corpus this library was measured on
```

and a user pack overrides whatever it names, **entry by entry**. Resolution order,
most specific first:

```
Config.patterns          a PatternSet, or a path to a file or directory
AUTOSXTRACT_PATTERNS     the same, from the environment
the locale pack          chosen from Config.language
base                     always underneath
```

This notebook writes a pack for a different domain, loads it all three ways, and
then shows the thing that makes the exercise worth doing: **a page that the
shipped pack calls a successful extraction and the new pack correctly refuses.**

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

logging.disable(logging.INFO)

from autosxtract import Config, patterns
from autosxtract.pdf.profile import PageProfile
from autosxtract.quality.gate import evaluate
from autosxtract.quality.metrics import domain_coverage
from autosxtract.quality.scoring import score_text
from autosxtract.quality.stamp import default as stamp_for

catalogue = patterns.default()
print("packs shipped   :", patterns.available_packs())
print("origins resolved:", catalogue.origins)
print("entries         :", len(catalogue.names()))
print()
print("a few names:", catalogue.names()[:8], "...")

## Every entry carries the measurement that fixed it

`why` is not a TOML comment: it is parsed into the `Entry` object, because it is
what a reader needs in order to decide whether *their* corpus invalidates the
entry. An entry whose `why` no longer holds is an entry to **delete**, not to
adjust in silence.

In [ ]:
for name in ("stamp.conformity", "metrics.glyph_index", "anchors.alphanumeric"):
    print(f"── {name}")
    print("  ", " ".join(catalogue.why(name).split())[:300])
    print()

## The six shapes an entry can take

Exactly one of `pattern`, `patterns`, `strings`, `words`, `map`, `text` decides the
kind, so a typo in a user pack is caught **at load** rather than at first use. Each
shape exists because a caller needs it in that form:

- `pattern` — one regex, optionally with `flags` and a literal `replacement`, so
  that a substitution's two halves travel together. Splitting them across a data
  file and a call site is how a pack ends up matching the OCR's corrupted currency
  symbol and writing back the wrong one.
- `patterns` — a list kept **uncompiled**, because the callers do different things
  with it: the stamp joins them into one alternation, the domain coverage reports
  *which* of them matched.
- `words` — a whitespace-separated block read as a set, written as running text on
  purpose: this list is meant to be edited by whoever adapts the library, and sixty
  quoted strings are not readable.

In [ ]:
print("patterns :", catalogue.patterns("metrics.domain")[:3], "...")
print("words    :", sorted(catalogue.words("metrics.stopwords"))[:12], "...")
print("strings  :", catalogue.strings("screening.identity_marks")[:4], "...")
print("map      :", dict(list(catalogue.mapping("prose.homoglyphs").items())[:4]), "...")
print("regex    :", catalogue.regex("metrics.glyph_index").pattern)
print()
# ``sub`` applies the entry's own replacement — the fix travels with the pattern.
print("sub      :", catalogue.sub("prose.wrong_currency_symbol", "o valor de RS 1.234,56 devidos"))

## Writing a pack

The pack below is for a **clinical records** corpus. It is still Portuguese — the
language layer (`metrics.stopwords`, `metrics.allowed_chars`) is fine — but the
*domain* is wrong in two specific places, and those are the two entries it
redefines:

- `metrics.domain`, the only scoring family that can **add** to a score. The
  shipped list is legal vocabulary; a clinical record contains none of it and is
  penalised for the absence.
- `stamp.conformity`, the boilerplate that has to come off before anything is
  measured. Every system that exports records prints its own footer, and it
  survives when the body of the page produces nothing.

Everything else — sixty-odd entries — is inherited. That is the whole argument for
entry-level merging: file-level merging would force whoever wants a different stamp
to copy the other sixty patterns, and **a copy is a fork that stops receiving
fixes**.

Two syntax notes. The pack uses TOML *literal* strings (`'...'` and `'''...'''`), which
carry a regex verbatim — no escaping layer between what is written and what `re`
compiles, which is exactly the defect a JSON catalogue would introduce. And the
Python string below is a **raw** string for the same reason, one level up.

In [ ]:
CLINICAL_PACK = r"""
# A pattern pack for clinical records — written for notebook 07.
#
# It redefines two entries and inherits the rest of base.toml and pt_br.toml.

[metrics.domain]
why = '''
The vocabulary a clinical record is expected to contain. The bundled list is legal
vocabulary, and a record holding none of it is penalised 0.15 for an absence that means
nothing here. This is the only scoring family that can ADD to a score.
'''
patterns = [
    '\bprontu[aá]rio\b',
    '\banamnese\b',
    '\bposologia\b',
    '\breceitu[aá]rio\b',
    '\bevolu[cç][aã]o cl[ií]nica\b',
    '\bhip[oó]tese diagn[oó]stica\b',
    '\bconduta\b',
]

[stamp.conformity]
why = '''
The footer this record system prints on every exported page. It does the same damage as
the court banner it replaces: it sits in an intact encoding and survives when the body of
the page produces nothing, so an extraction that returned only the footer measures as a
success unless it is stripped first.
'''
patterns = [
    'Documento gerado pelo sistema de prontu[aá]rio eletr[oô]nico.{0,120}',
    'Impresso em \d{2}/\d{2}/\d{4}.{0,60}',
]
""".strip()

pack_dir = Path(tempfile.mkdtemp(prefix="autosxtract-pack-"))
(pack_dir / "clinical.toml").write_text(CLINICAL_PACK, encoding="utf-8")
print("pack written to", pack_dir / "clinical.toml")

## Way 1 — `Config(patterns=...)`

A `PatternSet`, or a path to a file **or a directory** whose `*.toml` are merged in
sorted order — so a pack can be split by concern the way the bundled one is.

`pattern_set()` is a method rather than a field for the same reason the parallelism
knobs resolve in methods: the machine that reads the pack may not be the one that
serialised the configuration, and a path resolved at construction would point at a
file that is not there.

In [ ]:
resolved = Config(patterns=pack_dir).pattern_set()

print("origins :", resolved.origins)
print("entries :", len(resolved.names()), "(the bundled ones are still all there)")
print()
print("REPLACED  metrics.domain     :", resolved.patterns("metrics.domain")[:3], "...")
print("INHERITED metrics.stopwords  :", len(resolved.words("metrics.stopwords")), "words from pt_br")
print("INHERITED metrics.glyph_index:", resolved.regex("metrics.glyph_index").pattern, "from base")
print()
print("why, as the new pack states it:")
print(" ", " ".join(resolved.why("metrics.domain").split())[:220])

## Way 2 — the `AUTOSXTRACT_PATTERNS` environment variable

The same thing, for a process that must not be recompiled to change corpus — a
container, a batch job, a CI matrix. It sits *below* `Config.patterns`, so an
explicit configuration still wins.

The `reset()` call is the part worth remembering. Every pack is cached
(`functools.cache`) and **nothing invalidates it on its own**, because a catalogue
that reloaded itself mid-batch would classify the first half of the documents by one
rule and the second half by another. So: change the environment or edit a pack on
disk, then `patterns.reset()`.

In [ ]:
os.environ[patterns.ENVIRONMENT_VARIABLE] = str(pack_dir)
patterns.reset()
print("with the variable set   :", Config().pattern_set().origins)

del os.environ[patterns.ENVIRONMENT_VARIABLE]
patterns.reset()
print("with the variable unset :", Config().pattern_set().origins)

## Way 3 — `Config.language`

A language tag selects a **bundled** pack: `pt-BR` finds `pt_br`, and `pt` would find
`pt` if it existed. A language with no bundled pack falls back to the default one, and
that is deliberate rather than lenient — before the catalogue existed the patterns were
Portuguese whatever `Config.language` said, and raising here would break a
configuration that works today.

So the honest demonstration of way 3 on a stock install is the fallback itself.
Shipping a pack into `autosxtract/patterns/data/`, or pointing `AUTOSXTRACT_PATTERNS`
at one, is how that fallback is left behind.

In [ ]:
for tag in ("pt-BR", "pt", "es-ES", "en", None):
    print(f"  language={str(tag):<8} -> bundled pack {patterns.pack_for_language(tag)!r}")

print()
spanish = Config(language="es-ES")
print("Config(language='es-ES').pattern_set().origins ->", spanish.pattern_set().origins)
print("  ...which is the pt_br pack. The tag changed the OCR language request,")
print("  not the catalogue, because no es pack is bundled.")

## The payoff, part 1: a domain term the shipped pack misses

`metrics.domain` is scored as *coverage* — the fraction of the expected patterns that
appear — and it is the only family that can add to a score: below 0.03 it subtracts
0.15, above 0.10 it adds 0.05. On a clinical record the legal list matches nothing, so
a perfectly good page is penalised for an absence that means nothing.

`patterns.use()` installs a catalogue as the **process** default. The quality modules
take no configuration — that is the property that lets one criterion judge every step —
so they read this one rather than being handed a set per call.

In [ ]:
RECORD = (
    "PRONTUARIO ELETRONICO\n\n"
    "Anamnese: paciente refere dor abdominal ha tres dias, sem febre e sem "
    "vomitos. Hipotese diagnostica de gastrite. Conduta: solicitado exame "
    "laboratorial e prescrito receituario com posologia de um comprimido a "
    "cada oito horas. Evolucao clinica sera reavaliada em sete dias.\n"
    "Documento gerado pelo sistema de prontuario eletronico da unidade. "
    "Impresso em 04/02/2021 as 10h32."
)

for label, install in (("bundled pt-BR pack", None), ("clinical pack", resolved)):
    patterns.use(install)
    coverage = domain_coverage(RECORD)
    assessment = score_text(RECORD)
    print(f"{label:<20} domain terms matched: {coverage['count']}/"
          f"{len(patterns.default().patterns('metrics.domain'))}"
          f"   score {assessment['score']}")
    for reason in assessment["reasons"]:
        print(f"{'':<20}   - {reason}")
patterns.use(None)

## The payoff, part 2: the false success, reproduced for a new corpus

This is the failure the stamp entry exists to prevent, and it is corpus-specific by
construction. In an audit of 1,339 court documents, **227 extractions looked
successful and all there was, was the conformity banner** — 250 to 600 characters of
boilerplate that sail past any size threshold.

A clinical record system prints its own boilerplate, and the shipped pack has never
heard of it. Below is a page whose only surviving text is that footer, put through the
same acceptance gate twice.

In [ ]:
FOOTER = (
    "Documento gerado pelo sistema de prontuario eletronico da unidade de "
    "saude. Impresso em 04/02/2021 as 10h32.\n"
)
footer_only = FOOTER * 4
sheet = PageProfile(pages=1, has_image=True)

for label, install in (("bundled pt-BR pack", None), ("clinical pack", resolved)):
    patterns.use(install)
    stripped = stamp_for().strip(footer_only)
    verdict = evaluate(footer_only, sheet)
    print(f"{label}")
    print(f"  characters left after stripping: {len(stripped):>3} of {len(footer_only)}")
    print(f"  escalate: {verdict.escalate}   reason: {verdict.reason}")
    print()
patterns.use(None)

The bundled pack calls 436 characters of pure boilerplate an **adequate
extraction**. The clinical pack strips it and the gate refuses the page, which sends
it to the next step — which is the entire point of having a gate.

Nothing in the measurement code changed between those two runs. Everyone measures
through the same `StampStripper`, so replacing the patterns is all it takes.

## A pack that is wrong fails at load, not on document 4,000

`resolve()` compiles the whole catalogue — about sixty patterns, under a millisecond
— precisely so that a malformed user pack surfaces **at the top of the run, naming
the entry and the file**, instead of in the middle of a batch.

In [ ]:
from autosxtract.exceptions import InvalidConfiguration

BROKEN = r"""
[metrics.domain]
why = 'a character class nobody closed'
patterns = ['\bprontu[aario\b']
""".strip()

TYPO = r"""
[stamp.conformity]
why = 'two kinds declared in one entry'
pattern = 'Impresso em'
patterns = ['Impresso em']
""".strip()

for label, source in (("bad regex", BROKEN), ("two kinds", TYPO)):
    bad_dir = Path(tempfile.mkdtemp(prefix="autosxtract-bad-"))
    (bad_dir / "broken.toml").write_text(source, encoding="utf-8")
    try:
        Config(patterns=bad_dir).pattern_set()
    except InvalidConfiguration as exc:
        print(f"{label:<11} -> InvalidConfiguration")
        print(f"{'':<14}{exc}")
    print()

Missing entries fail the same way, and the message says the thing a reader actually
needs to know — that a pack **overrides** entries and never removes them.

In [ ]:
try:
    resolved.patterns("metrics.no_such_entry")
except InvalidConfiguration as exc:
    print(exc)

print()
# Asking for an entry as the wrong kind is caught too, naming the file it came from.
try:
    resolved.regex("metrics.domain")     # it is a ``patterns`` list, not a single regex
except InvalidConfiguration as exc:
    print(exc)

## Checklist for a pack of your own

1. **Redefine only what your corpus actually changes.** Every entry you do not name is
   inherited and keeps receiving fixes; every entry you copy is a fork.
2. **Write the `why`.** It is parsed, it is what `patterns.why(name)` returns, and it is
   what tells the next reader whether the entry still holds. State the measurement, not
   the intention.
3. **Use TOML literal strings** (`'...'`, `'''...'''`) for regexes, so no escaping layer
   sits between what you wrote and what `re` compiles.
4. **Start with `stamp.conformity` and `metrics.domain`.** They are the two entries that
   silently distort every measurement in a new corpus — one manufactures false
   successes, the other penalises legitimate text.
5. **Then look at `metrics.stopwords` and `metrics.allowed_chars`** if the *language*
   changes rather than only the domain. Those two are what decide "is this even
   readable text", and they are written as running text precisely so they can be edited.
6. **Call `patterns.reset()`** after editing a pack on disk or changing the environment.
   Nothing reloads on its own, on purpose.
7. **Measure at the cascade level** (notebook 04). A pack changes what the gates think,
   and what the gates think changes which steps run.

## The end of the tour

You have now seen the whole library from the outside: the cascade and its provenance,
each step by hand, both gates, the configuration and how to measure a change to it, and
all three extension points — an engine, a step, and the pattern catalogue. Everything
here ran on a machine with no Apple hardware and no real document, which is the only
kind of documentation this project keeps.